In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


In [2]:
from urllib.request import urlopen
from bs4 import BeautifulSoup as bs
import requests
import json
import datetime
import time
import re
import tqdm


In [3]:
def page_number(start, end):
    """
    Description:
        Function to generate a list of webpage address for a given page number

    Parameters:
        start (int) : starting page number
        end (int)   : ending page number
    Returns:
        a list of listing web address 
    
    """
    
    page_url = 'https://www.carlist.my/cars-for-sale/perodua/myvi/malaysia?page_number='
    list_page = []
    for i in range(start,end+1):
        list_page.append(page_url+str(i))
    return list_page

page_number(2,4)

['https://www.carlist.my/cars-for-sale/perodua/myvi/malaysia?page_number=2',
 'https://www.carlist.my/cars-for-sale/perodua/myvi/malaysia?page_number=3',
 'https://www.carlist.my/cars-for-sale/perodua/myvi/malaysia?page_number=4']

In [86]:
def extract_price_and_url(text):
 
    # Pattern for RM price
    price_pattern = r'RM\s*([0-9,]+)'
    
    # Pattern for URL
    url_pattern = r'(https://www\.carlist\.my/[\w-]+/[^\s]+?)\.?(?=\s|$)'
    
    # Extract price
    price_match = re.search(price_pattern, text)
    price = price_match.group(1).replace(' ', '') if price_match else None
    
    # Extract URL
    url_match = re.search(url_pattern, text)
    url = url_match.group(1) if url_match else None
    
    return price, url


In [88]:
indv_url = 'https://www.carlist.my/used-cars/2022-perodua-myvi-1-3-g-promo-siap-otr-warranty-5thn-mileage-4k-shj-8228-plate-no-full-servis-rekod/12061552'

def extract_location_and_date(indv_url):
    
    article1_data = {
        'location': None,
        'state': None
    } 
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36"}
        webpage = requests.get(url = indv_url, headers= headers)
        soup2 = bs(webpage.text, 'html.parser')
        article1 = soup2.find('script', {'type': 'application/ld+json'})
                
        if article1 and article1.string: 
            
            try: 
                json_data = json.loads(article1.string)
                if isinstance(json_data,list) and len(json_data) > 0:
                    try:
                        car_data = json_data[0]["offers"]['seller']['homeLocation']['address']
                        article1_data.update({ 
                            'location': car_data.get('addressLocality'),
                            'state': car_data.get('addressRegion')
                            })
                    except (KeyError, TypeError) as e:
                        print(f'Error accessing JSON: {e}')
            except json.JSONDecodeError as e:
                print(f'Error parsing JSON: {e}')
        '''
        else:
            print("No JSON-LD Data found")
            #test = 0 '''
        #time.sleep(2)
        return article1_data
    
    except:
        print(f'Error in extract_location_and_date: {e}')
        return article1_data

#print(test)
extract_location_and_date(indv_url)

{'location': 'Jln Ampang', 'state': 'Kuala Lumpur'}

In [82]:
page_url = 'https://www.carlist.my/cars-for-sale/perodua/myvi/malaysia?page_number=2'

def main_extraction(page_url):
    headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36"}

    page = requests.get(url = page_url, headers=headers)
    soup = bs(page.text, 'html.parser')

    articles = soup.findAll('article',{
            'class': ['js--listing', 'article--details']
        })

    if articles:

        #print(f"Type of find_all() result: {type(articles)}")  # ResultSet object
        print(f"Number of articles found: {len(articles)}\n")

        results = []

        for article in articles:
            try:
                
                article_data = {
                    'listing_id': article.get('data-listing-id'),
                    'title': article.get('data-title'),
                    'installment': article.get('data-installment'),
                    'year': article.get('data-year', ''),
                    'variant': article.get('data-variant', ''),
                    'mileage': article.get('data-mileage', ''),
                    'transmission': article.get('data-transmission', ''),
                    'location': None,
                    'state': None,
                    'image': article.get('data-image-src')

                }

                default_text = article.get('data-default-line-text', '')
                price, url = extract_price_and_url(default_text)
                article_data['price'] = price
                article_data['url'] = url

                if url:
                    location_data = extract_location_and_date(url)

                    if isinstance(location_data,dict):
                        for key, value in location_data.items():
                            if value is not None:
                                article_data[key] = value
                
                results.append(article_data)
                
            except Exception as e:
                print(f"Error processing article {article.get('data-listing-id', 'unknown')}: {str(e)}")
                continue  # Skip this article and continue with next one
        
        if results:
            df = pd.DataFrame(results)
            # Replace empty strings and None values with pd.NA
            df = df.replace(['', None], pd.NA)
            #print(df)
            return df
        else:
            print("No results were successfully processed")

#main_extraction(page_url)['location'].isna()

In [90]:
def main(start,end):
    page_list = page_number(start,end)
    all_df = pd.DataFrame([])

    try:
        for index,item in enumerate(page_list):
            results = main_extraction(item)
            
            if len(results) > 0:
                 all_df = pd.concat([all_df,results])

                #print(f"Processed item {index + 1}/{len(item_list)}: Found {len(df)} entries")
            else:
                print(f"Processed item {index + 1}: No entries found")
        '''
        if all_df:
                final_df = pd.concat(all_df, ignore_index=True)
                print(f"\nTotal entries in combined DataFrame: {len(final_df)}")
                return final_df
        else:
            print("No data was processed successfully")
            return pd.DataFrame() '''
            
    except Exception as e:
        print(f"Error in main processing: {str(e)}")
        return pd.DataFrame()
    
    return all_df
    
combined_df = main(3,4)
#print(combined_df['location'].isna().sum())
   



Number of articles found: 25

Number of articles found: 25



In [91]:
print(len(combined_df))

date = pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
combined_df.to_csv(f'C:/Users/asyak/Downloads/Page3to7({date}).csv',index = False)
print("Data has been succesfully scrapped and saved to CSV file")


50
Data has been succesfully scrapped and saved to CSV file
